任务1：图像匹配与拼接

In [ ]:
import sys
import cv2
import numpy as np

def compute_homography(src_pts, dst_pts):
    
    """ 请在此处开始编写你的代码：
    
    计算单应矩阵 H。
    
    :param src_pts: 源点坐标
    :param dst_pts: 目标点坐标
    :return: 单应矩阵 H

   将源点坐标(src_pts)变换到目标点坐标(dst_pts)：

    1. 根据输入的源点坐标和目标点坐标,构建一个线性方程组。每对对应的源点和目标点可以提供两个方程。

       其中(x, y)是源点坐标,(u, v)是目标点坐标,h11到h33是单应矩阵H的9个元素。

    2. 将所有点对产生的方程系数存储在矩阵A中。A的每一行对应一个方程的系数。

    3. 使用奇异值分解(SVD)求解方程组Ah=0的解,其中h是单应矩阵H的9个元素组成的列向量。SVD分解得到的矩阵V的最后一列对应最小奇异值,即为方程组的解。

    4. 将解向量重塑为3x3矩阵,并将其除以H[3,3]进行归一化处理,得到最终的单应矩阵H。

    总的来说,通过构建线性方程组并使用SVD求解,得到了将源点变换到目标点的单应矩阵H。单应矩阵H可用于对图像进行变换,实现图像配准、拼接等操作。
    
    """
    num_points = src_pts.shape[0]
    A = np.zeros((2 * num_points, 9)) # 初始化线性方程组矩阵
    
    for i in range(num_points):
        x, y = src_pts[i]
        u, v = dst_pts[i]
        A[2 * i] = [-x, -y, -1, 0, 0, 0, u * x, u * y, u] # 对应于 x 的变换方程
        A[2 * i + 1] = [0, 0, 0, -x, -y, -1, v * x, v * y, v] # 对应于 y 的变换方程
        
    _, _, Vt = np.linalg.svd(A)
    H = Vt[-1].reshape(3, 3) # 取最小奇异值对应的奇异向量，重塑为3x3矩阵
    H = H / H[2, 2]
    
    return H

def ransac_homography(src_pts, dst_pts, threshold=5.0, max_iter=2000, confidence=0.99):
    
    """ 请在此处开始编写你的代码：
    
    使用RANSAC估计单应矩阵。
    
    :param src_pts: 源点坐标
    :param dst_pts: 目标点坐标
    :param threshold: 内点距离阈值
    :param max_iter: 最大迭代次数
    :param confidence: 置信度
    :return: 最佳单应矩阵 H 和内点掩码

    这个代码模块使用RANSAC算法估计单应矩阵,其思路如下:

    1. RANSAC算法通过反复随机抽样和估计模型参数,从包含异常值的数据中求解最优模型。

    2. 在每次迭代中,随机选择4对源点和目标点,用这4对点计算出一个单应矩阵H。

    3. 使用计算出的单应矩阵H将所有源点变换到目标点的坐标系,并计算变换后的源点与实际目标点之间的距离(残差)。

    4. 根据残差和给定的阈值,判断每个点是否为内点(即符合当前估计的单应矩阵的点)。同时更新最佳单应矩阵和对应的内点数量。

    5. 重复步骤2-4,直到达到最大迭代次数或者内点数量占总点数的比例超过给定的置信度。最终返回最佳单应矩阵和内点掩码。

    通过RANSAC算法,可以在存在噪声和异常值的情况下,鲁棒地估计出最优的单应矩阵,用于图像配准和拼接任务。
    
    """
    num_pts = src_pts.shape[0]
    best_inliers = None
    best_H = None
    iteration = 0
    best_inlier_count = 0
    
    while iteration < max_iter:
        indices = np.random.choice(num_pts, 4, replace=False)
        src_sample = src_pts[indices]
        dst_sample = dst_pts[indices]
        
        H_candidate = compute_homography(src_sample, dst_sample)
        
        src_homog = np.column_stack((src_pts, np.ones(num_pts))) #  将 src_pts 的 x,y 坐标和一个全1的列向量堆叠起来
        dst_homog_pred = np.dot(H_candidate, src_homog.T).T
        dst_homog_pred /= dst_homog_pred[:, 2][:, np.newaxis]  # 将变换后的齐次坐标 𝑥′,𝑦′,𝑤′ 归一化，使得每个点的第三个坐标为1
        
        # 计算变换后的点与实际目标点之间的欧几里得距离
        errors = np.sqrt((dst_homog_pred[:, 0] - dst_pts[:, 0]) ** 2 + (dst_homog_pred[:, 1] - dst_pts[:, 1]) ** 2)
        inliers = errors < threshold
        inlier_count = np.sum(inliers)
        
        if inlier_count > best_inlier_count:
            best_inliers = inliers
            best_H = H_candidate
            best_inlier_count = inlier_count
            
            # Update the number of iterations needed for a given confidence level
            inlier_ratio = best_inlier_count / num_pts  # 内点比例
            n_estimated = np.log(1 - confidence) / np.log(1 - inlier_ratio ** 4)
            max_iter = min(max_iter, int(n_estimated))
        
        iteration += 1

    if best_H is not None:
        return best_H, best_inliers.astype(int)
    else:
        return None, None

def Panorama_stitching(image_right, image_left, downscale_factor=2):
    """
    全景拼接函数。
    
    :param image_right: 右侧图像
    :param image_left: 左侧图像
    :param downscale_factor: 下采样因子
    :return: 拼接后的全景图像
    
    """
    # 对图像进行下采样，以期减少计算量
    image_right_downscaled = cv2.resize(image_right, None, fx=1/downscale_factor, fy=1/downscale_factor)
    image_left_downscaled = cv2.resize(image_left, None, fx=1/downscale_factor, fy=1/downscale_factor)

    # 转换为灰度图像
    gray_right = cv2.cvtColor(image_right_downscaled, cv2.COLOR_BGR2GRAY)
    gray_left = cv2.cvtColor(image_left_downscaled, cv2.COLOR_BGR2GRAY)

    # 使用SIFT特征检测器
    sift = cv2.SIFT_create()
    keypoints_right, descriptors_right = sift.detectAndCompute(gray_right, None)  #接受一个灰度图像和一个掩码
    keypoints_left, descriptors_left = sift.detectAndCompute(gray_left, None)
    # 每个关键点都是一个带有许多属性的对象（如位置、尺度、方向等）
    # 一个数组，每行对应一个关键点的描述符

    # 使用暴力匹配器进行特征匹配
    bf = cv2.BFMatcher()
    matches = bf.knnMatch(descriptors_right, descriptors_left, k=2)

    # 通过Lowe's ratio test筛选出好的匹配点
    good_matches = []
    for m, n in matches:
        if m.distance < 0.7 * n.distance:
            good_matches.append(m)

    if len(good_matches) > 10:
        src_pts = np.float32([keypoints_right[m.queryIdx].pt for m in good_matches]) * downscale_factor
        dst_pts = np.float32([keypoints_left[m.trainIdx].pt for m in good_matches]) * downscale_factor

        # 使用RANSAC计算单应矩阵
        H, mask = ransac_homography(src_pts, dst_pts)
        matches_mask = np.array(mask, dtype=int).tolist()  # 转换为Int类型列表

        # 图像变形——透视变换
        h_right, w_right = image_right.shape[:2]
        h_left, w_left = image_left.shape[:2]
        panorama = cv2.warpPerspective(image_right, H, (w_right + w_left, max(h_right, h_left)))
        panorama[0:h_left, 0:w_left] = image_left

        # 显示单应矩阵
        print("Homography matrix:")
        print(H)

        # 显示匹配点比例
        print(f"Matches ratio: {len(good_matches) / len(matches):.2f}")

        # 显示匹配点
        draw_params = dict(matchColor=(0, 255, 0),  # 在匹配的关键点间画绿线
                           singlePointColor=None,
                           matchesMask=matches_mask,  # 只画内点
                           flags=2)
        img_matches = cv2.drawMatches(image_right_downscaled, keypoints_right, image_left_downscaled, keypoints_left, good_matches, None, **draw_params)

        # cv2.imshow("Matches", img_matches)
        cv2.imwrite('matches.jpg', img_matches)

        return panorama

    else:
        print("Not enough matches are found - {}/{}".format(len(good_matches), 10))
        return None

# 主程序
if (__name__ == "__main__"):
    # 读取左右图像
# 正确写法：前面加 r 代表原始字符串，不转义
    image_right = cv2.imread(r'right.jpg')
    image_left = cv2.imread(r'left.jpg')

    # 调用全景拼接函数，得到全景图像
    panorama = Panorama_stitching(image_right, image_left)

    # 判断全景图像是否为空，如果不为空则显示并保存全景图像
    if panorama is not None:
        # 显示全景图像
        # cv2.imshow('Panorama', panorama)
        cv2.imwrite('ppa09pa.jpg', panorama)  # 保存全景图像
        # cv2.waitKey(0)
    else:
        print("Panorama stitching failed.")

    # cv2.destroyAllWindows()
    sys.exit(0)

Homography matrix:
[[ 5.21716671e-01  4.58782018e-01  5.81554155e+02]
 [-2.78782070e-01  9.96111048e-01  2.08270520e+02]
 [-2.37177151e-04  1.48577649e-04  1.00000000e+00]]
Matches ratio: 0.21


SystemExit: 0

C:\Users\81349\AppData\Roaming\Python\Python314\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


任务2：车道线检测

In [ ]:
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt


# 灰度转换
def convert_to_grayscale(image):
    print("灰度转换已完成 10%")
    return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)


def gaussian_kernel(size, sigma=1.0):
    size = int(size) // 2
    x, y = np.mgrid[-size:size+1, -size:size+1]
    normal = 1 / (2.0 * np.pi * sigma**2)
    g = np.exp(-((x**2 + y**2) / (2.0*sigma**2))) * normal
    return g


# 滤波
def apply_gaussian_blur(image, kernel_size=5, sigma=1.0):
    print("高斯滤波已完成 20%")
    kernel = gaussian_kernel(kernel_size, sigma)
    output = np.zeros_like(image)

    # 添加边界填充，以便卷积时边缘像素也能正确处理
    pad_height = kernel_size // 2
    pad_width = kernel_size // 2
    padded_image = np.pad(image, [(pad_height, pad_height), (pad_width, pad_width)], mode='constant', constant_values=0)

    # 对图像的每个像素应用高斯核
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            region = padded_image[i:i + kernel_size, j:j + kernel_size]
            output[i, j] = np.sum(region * kernel)

    return output


# 边缘检测
def sobel_filters(image):
    Kx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
    Ky = np.array([[1, 2, 1], [0, 0, 0], [-1, -2, -1]])
    Ix = np.convolve(image.flatten(), Kx.flatten(), 'same').reshape(image.shape)
    Iy = np.convolve(image.flatten(), Ky.flatten(), 'same').reshape(image.shape)
    G = np.sqrt(Ix**2 + Iy**2)
    Theta = np.arctan2(Iy, Ix)
    return G, Theta


def non_max_suppression(gradient, direction):
    M, N = gradient.shape
    Z = np.zeros((M, N))
    angle = direction * 180. / np.pi
    angle[angle < 0] += 180
    for i in range(1, M-1):
        for j in range(1, N-1):
            try:
                q = 255
                r = 255
                if (0 <= angle[i,j] < 45) or (135 <= angle[i,j] <= 180):
                    q = gradient[i, j+1]
                    r = gradient[i, j-1]
                elif (45 <= angle[i,j] < 135):
                    q = gradient[i+1, j]
                    r = gradient[i-1, j]

                if gradient[i,j] >= q and gradient[i,j] >= r:
                    Z[i,j] = gradient[i,j]
                else:
                    Z[i,j] = 0
            except IndexError:
                pass
    return Z


def threshold(image, low, high):
    strong = 255
    weak = 25
    result = np.zeros_like(image)
    strong_i, strong_j = np.where(image >= high)
    weak_i, weak_j = np.where((image >= low) & (image < high))
    result[strong_i, strong_j] = strong
    result[weak_i, weak_j] = weak
    return result


def detect_edges(image, low_threshold=50, high_threshold=150):
    print("边缘检测已完成 30%")
    gradient, direction = sobel_filters(image)
    non_max_img = non_max_suppression(gradient, direction)
    final_img = threshold(non_max_img, low_threshold, high_threshold)
    final_img = np.clip(final_img, 0, 255).astype(np.uint8)
    return final_img


# 提取感兴趣区域ROI：一开始尝试定义为图像下半部分的一个三角形区域。
# 改进后，ROI被定义成一个更符合实际车道形状的梯形区域。
def region_of_interest(image):
    height = image.shape[0]
    width = image.shape[1]
    polygons = np.array([
        [
            (int(width * 0.1), height),
            (int(width * 0.9), height),
            (int(width * 0.55), int(height * 0.6)),
            (int(width * 0.45), int(height * 0.6))
        ]
    ])
    mask = np.zeros_like(image)
    cv2.fillPoly(mask, polygons, 255)
    masked_image = cv2.bitwise_and(image, mask)
    print("提取感兴趣区域已完成 40%")
    return masked_image


# 霍夫变换检测直线：通过降低 minLineLength （从100降到20）和增加 maxLineGap （从50增加到300）等，
def hough_transform(image, theta_res=1, rho_res=1, threshold=20):
    height, width = image.shape
    max_dist = int(np.hypot(height, width))
    rhos = np.arange(-max_dist, max_dist, rho_res)
    thetas = np.radians(np.arange(-90, 90, theta_res))

    cos_t = np.cos(thetas)
    sin_t = np.sin(thetas)
    num_thetas = len(thetas)

    accumulator = np.zeros((2 * max_dist, num_thetas), dtype=np.int32)
    y_idxs, x_idxs = np.nonzero(image)

    for i in range(len(x_idxs)):
        x = x_idxs[i]
        y = y_idxs[i]
        for t_idx in range(num_thetas):
            rho = int(round(x * cos_t[t_idx] + y * sin_t[t_idx]) + max_dist)
            accumulator[rho, t_idx] += 1

    lines = []
    for r_idx in range(accumulator.shape[0]):
        for t_idx in range(accumulator.shape[1]):
            if accumulator[r_idx, t_idx] > threshold:
                rho = rhos[r_idx]
                theta = thetas[t_idx]
                a = np.cos(theta)
                b = np.sin(theta)
                x0 = a * rho
                y0 = b * rho
                x1 = int(x0 + 1000 * (-b))
                y1 = int(y0 + 1000 * (a))
                x2 = int(x0 - 1000 * (-b))
                y2 = int(y0 - 1000 * (a))
                lines.append([x1, y1, x2, y2])

    return np.array(lines).reshape(-1, 1, 4)


# 线段后处理：对检测到的线段进行后处理，计算左右车道线的平均斜率和截距；
# 并对检测到的线段进行合并，减少了由于噪声和断裂导致的检测不连续性，使检测结果更加稳定和连续。
def average_slope_intercept(image, lines):
    left_lines = []  # (slope, intercept)
    right_lines = []  # (slope, intercept)
    left_weights = []  # length of the line segment
    right_weights = []  # length of the line segment

    for line in lines:
        for x1, y1, x2, y2 in line:
            if x1 == x2: # 两个端点的 x 坐标相同，表示这是一条垂直线。跳过
                continue  # skip vertical lines
            slope = (y2 - y1) / (x2 - x1) # 斜率
            intercept = y1 - slope * x1 # 截距
            length = np.sqrt((y2 - y1) ** 2 + (x2 - x1) ** 2) # 长度
            if slope < 0:  # left lane 左
                left_lines.append((slope, intercept))
                left_weights.append(length)
            else:  # right lane 右
                right_lines.append((slope, intercept))
                right_weights.append(length)
    
    # 加权平均的方法来确定每侧车道的代表性直线
    # 每条线的 (slope, intercept) 与其长度作为权重相乘，然后除以所有权重的和
    left_lane = np.dot(left_weights, left_lines) / np.sum(left_weights) if len(left_weights) > 0 else None
    right_lane = np.dot(right_weights, right_lines) / np.sum(right_weights) if len(right_weights) > 0 else None
    print("线段后处理已完成 80%")
    return left_lane, right_lane


def make_line_points(y1, y2, line):
    if line is None:
        return None
    slope, intercept = line
    x1 = int((y1 - intercept) / slope)
    x2 = int((y2 - intercept) / slope)
    y1 = int(y1)
    y2 = int(y2)
    return ((x1, y1), (x2, y2))


def draw_lines(image, lines):
    line_image = np.zeros_like(image)
    if lines is not None:
        left_lane, right_lane = average_slope_intercept(image, lines)
        y1 = image.shape[0]
        y2 = y1 * 0.6
        left_line = make_line_points(y1, y2, left_lane)
        right_line = make_line_points(y1, y2, right_lane)
        for line in [left_line, right_line]:
            if line is not None:
                cv2.line(line_image, *line, (0, 255, 0), 10)  # 绿色线条
    print("车道线绘制已完成 90%")
    return line_image


# 处理单帧图像
def process_image(image):
    gray = convert_to_grayscale(image)
    blurred = apply_gaussian_blur(gray)
    edges = detect_edges(blurred)
    roi = region_of_interest(edges)
    lines = hough_transform(roi)
    line_image = draw_lines(image, lines)
    combined = cv2.addWeighted(image, 0.8, line_image, 1, 1)
    print("单帧图像处理已完成 100%")
    return combined


# 处理视频
def process_video(input_video_path, output_video_path):
    cap = cv2.VideoCapture(input_video_path)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, 20.0, (int(cap.get(3)), int(cap.get(4))))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        processed_frame = process_image(frame)
        out.write(processed_frame)
        print("视频处理进度: {:.2f}%".format((cap.get(cv2.CAP_PROP_POS_FRAMES) / cap.get(cv2.CAP_PROP_FRAME_COUNT)) * 100))
        print()

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print("视频处理已完成 100%")


if (__name__ == '__main__'):
    # 处理视频
    process_video('drive.mp4', 'output_drive_V3.mp4')
    sys.exit(0)

任务3：哈里斯角点检测

In [19]:
import cv2
import numpy as np

def harris_corner_detection(image_path, threshold_ratio=0.01, block_size=2, ksize=3, k=0.04):
    """
    哈里斯角点检测（增强版）
    :param image_path: 图像路径
    :param threshold_ratio: 角点响应阈值比例（相对于最大值），越小角点越多
    :param block_size: 邻域大小
    :param ksize: Sobel算子孔径大小
    :param k: Harris自由参数
    """
    # 1. 读取图像
    img = cv2.imread(image_path)
    if img is None:
        print(f"错误：无法读取图像 {image_path}")
        return
    img_copy = img.copy()

    # 2. 转为灰度图并转为float32
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray_float = np.float32(gray)

    # 3. 哈里斯角点检测
    dst = cv2.cornerHarris(gray_float, block_size, ksize, k)
    dst = cv2.dilate(dst, None)  # 膨胀使角点更明显

    # 4. 阈值筛选
    threshold = threshold_ratio * dst.max()
    # 获取满足条件的像素坐标（y, x）
    corners = np.argwhere(dst > threshold)

    # 5. 非极大值抑制（NMS）—— 避免同一角点附近多个响应
    # 创建一个与原图大小相同的掩码，标记已选中的角点区域
    nms_radius = 5  # 抑制半径（像素）
    nms_mask = np.zeros_like(gray, dtype=bool)
    filtered_corners = []
    # 按响应强度降序排序（响应值越大越可能是真角点）
    corner_responses = [(y, x, dst[y, x]) for y, x in corners]
    corner_responses.sort(key=lambda p: p[2], reverse=True)

    for y, x, resp in corner_responses:
        # 检查该点周围是否已有被选中的角点
        y_min = max(0, y - nms_radius)
        y_max = min(gray.shape[0], y + nms_radius + 1)
        x_min = max(0, x - nms_radius)
        x_max = min(gray.shape[1], x + nms_radius + 1)
        if not np.any(nms_mask[y_min:y_max, x_min:x_max]):
            filtered_corners.append((x, y))  # 存储为 (x, y) 方便绘图
            nms_mask[y, x] = True

    # 6. 在原图上绘制角点（彩色圆圈）
    for (x, y) in filtered_corners:
        cv2.circle(img_copy, (x, y), 4, (0, 0, 255), -1)  # 红色实心圆
        # 可选：绘制十字标记
        # cv2.drawMarker(img_copy, (x, y), (0, 255, 0), cv2.MARKER_CROSS, 8, 2)

    # 7. 显示结果
    print(f"检测到角点数量：{len(filtered_corners)}")
    cv2.imshow("Original Image", img)
    cv2.imshow("Harris Corners (NMS + Circles)", img_copy)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    # 可选：保存结果
    # cv2.imwrite("harris_result_with_circles.jpg", img_copy)

    return filtered_corners  # 返回角点坐标列表，供后续使用


if __name__ == "__main__":
    # 使用示例：请将路径改为你的图片路径
    image_path = "left.jpg"  # 替换为实际路径
    corners = harris_corner_detection(image_path, threshold_ratio=0.02, block_size=2, ksize=3, k=0.04)
    print("角点坐标列表（前10个）：", corners[:10])

检测到角点数量：4075
角点坐标列表（前10个）： [(np.int64(1075), np.int64(1515)), (np.int64(2051), np.int64(1094)), (np.int64(1127), np.int64(1098)), (np.int64(2083), np.int64(1096)), (np.int64(2040), np.int64(1098)), (np.int64(1445), np.int64(1060)), (np.int64(1613), np.int64(979)), (np.int64(1065), np.int64(1165)), (np.int64(1489), np.int64(1681)), (np.int64(2072), np.int64(1100))]
